In [2]:
%pip install pandas scikit-learn
%pip install --upgrade kagglehub
import os
import pandas as pd



import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline

from sklearn.metrics import classification_report, confusion_matrix

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 280.5 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 2.5 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


/home/annika/cloud-computing-backend/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Import data
print("\n \n ")
path = kagglehub.dataset_download("andrewmvd/trip-advisor-hotel-reviews")
print("Path to dataset files:", path)
filedir = os.path.join(path, os.listdir(path)[0])

100%|██████████| 5.14M/5.14M [00:01<00:00, 3.95MB/s]

Extracting files...


Path to dataset files: /home/annika/.cache/kagglehub/datasets/andrewmvd/trip-advisor-hotel-reviews/versions/2


In [4]:
df = pd.read_csv(filedir)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20491 entries, 0 to 20490
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Review  20491 non-null  str  
 1   Rating  20491 non-null  int64
dtypes: int64(1), str(1)
memory usage: 320.3 KB


In [5]:
def review_to_sentiment(review):
  if review < 3:
    return "NEG"
  if review >= 3 and review < 5:
    return "NEU"
  if review >=5:
    return "POS"
df["sentiment"] = df["Rating"].apply(review_to_sentiment)

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

# 1. Define the pipeline with the classifier as the final step
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('svc', LinearSVC()) # This is allowed as the LAST step
])

# 2. Wrap the WHOLE pipeline in CalibratedClassifierCV to get confidence scores
model = CalibratedClassifierCV(pipeline, cv=5) 


In [8]:
df_sampled = df.groupby("sentiment").sample(n=4000, replace=True).reset_index(drop=True)
X = df_sampled["Review"]
y = df_sampled["sentiment"]
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state = 40)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         NEG       0.92      0.95      0.94       802
         NEU       0.77      0.71      0.74       811
         POS       0.78      0.81      0.79       787

    accuracy                           0.82      2400
   macro avg       0.82      0.82      0.82      2400
weighted avg       0.82      0.82      0.82      2400



In [22]:
sentences = ["I love it!"]
print("Sentiment: ", model.predict(sentences))
print("Confidence: ", model.predict_proba(sentences))

def get_confidence(model, sentence):
    try:
        probabilities = model.predict_proba(sentence)[0]

        # Use max() to get the confidence score regardless of the label order
        confidence = float(probabilities.max())

        return confidence
    except Exception as e:
        print(f"Error calculating confidence: {e}")
        return None

confi = get_confidence(model, sentences)
print(f"Calculated confidence: {confi}")

Sentiment:  ['POS']
Confidence:  [[0.0501411  0.33862654 0.61123236]]
Calculated confidence: 0.6112323640109831


In [23]:
import pickle
with open("model.plk", "wb") as file:
  pickle.dump(model, file)